In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, TimestampType

# Preliminary Version.

spark = SparkSession.builder.appName("Bronze").getOrCreate()

schema = StructType(
    [
        StructField("observation_uuid", StringType(), nullable=False),
        StructField("observer_id", IntegerType(), nullable=True),
        StructField("latitude", DoubleType(), nullable=True),
        StructField("longitude", DoubleType(), nullable=True),
        StructField("positional_accuracy", IntegerType(), nullable=True),
        StructField("taxon_id", IntegerType(), nullable=True),
        StructField("quality_grade", StringType(), nullable=True),
        StructField("observed_on", TimestampType(), nullable=True),
        StructField("anomaly_score", DoubleType(), nullable=True)
    ]
)

metadata_df = spark.read.format("csv") \
    .schema(schema) \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .option("sep", "\t") \
    .load("s3a://inaturalist-open-data/observations.csv.gz")

max_lat = 18.5052
min_lat = 5.5810
max_long = 126.5374
min_long = 117.1743

filtered_df = metadata_df.filter(
    F.col("latitude").between(min_lat, max_lat)
    & F.col("longitude").between(min_long, max_long)
)

ingested_df = filtered_df.withColumn(
    "ingestion_timestamp",
    F.current_timestamp()
).withColumn(
    "source_file_path",
    F.col("_metadata.file_path")
)

ingested_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("philippinebiodiveristyplatform.bronze.observations")